In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 19.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! pip install scikeras

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, InputLayer, Conv1D, Flatten, LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DL models/df_cleaned.csv')

In [ ]:
df.head(2)

,Unnamed: 0,TARGET,EXT_SOURCE_MEAN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,BURO_AMT_CREDIT_SUM_DEBT_MEAN,BURO_AMT_CREDIT_SUM_MEAN,BURO_AMT_CREDIT_MAX_OVERDUE_MEAN,PREV_AMT_ANNUITY_MEAN,...,AGE,AGE_GROUP,YEARS_REGISTRATION,YEARS_ID_PUBLISH,YEARS_EMPLOYED,BURO_DAYS_CREDIT_ENDDATE_MEAN_YEARS,BURO_DAYS_CREDIT_MEAN_YEARS,BURO_DAYS_CREDIT_UPDATE_MEAN_YEARS,PREV_DAYS_DECISION_MAX_YEARS,INST_DAYS_ENTRY_PAYMENT_MAX_YEARS
0,0,1.0,0.1617,202500.0,406597.5,24700.5,49156.2,108131.945,1681.0289,9251.775,...,26,25-34,10,6,2.0,1.0,2.4,1.4,1.7,0.1
1,1,0.0,0.4668,270000.0,1293502.5,35698.5,0.0,254350.120,0.0000,56553.990,...,46,45-54,3,1,3.0,1.5,3.8,2.2,2.0,1.5


In [ ]:
df = df.drop(columns=['Unnamed: 0'])

In [ ]:
y = df['TARGET'].astype(int)
X = df.drop(columns=['TARGET'])

In [ ]:
# Preprocessing
cat_cols = [
    "ORGANIZATION_TYPE", "NAME_FAMILY_STATUS", "AGE_GROUP"
    # , "NAME_EDUCATION_TYPE", "NAME_INCOME_TYPE",
    # "NAME_FAMILY_STATUS", "ORGANIZATION_TYPE","WEEKDAY_APPR_PROCESS_START"
]
num_cols = [col for col in X.columns if col not in cat_cols]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
pca = PCA(n_components=0.95, random_state=SEED)
prep_pca = Pipeline([
    ("pre", preprocessor),
    ("pca", pca)
])

In [ ]:
# Train / Val / Test split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED)

X_train_pca = prep_pca.fit_transform(X_train)
X_val_pca = prep_pca.transform(X_val)
X_test_pca = prep_pca.transform(X_test)

In [ ]:
# SMOTE
smote = SMOTE(random_state=SEED)
X_train_pca, y_train = smote.fit_resample(X_train_pca, y_train)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import numpy as np

In [ ]:
input_dim = X_train_pca.shape[1]

In [ ]:
def build_mlp(units1=64, units2=32, dropout_rate=0.3, lr=1e-3):
    model = Sequential([
        InputLayer(shape=(input_dim,)),
        Dense(units1, activation='relu'),
        Dropout(dropout_rate),
        Dense(units2, activation='relu'),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

In [ ]:
mlp_keras_param = {
    "units1": [64],
    "units2": [32, 64],
    "dropout_rate": [0.2, 0.4],
    "lr": [1e-3, 1e-4],
    "batch_size": [32, 64]
}

In [ ]:
def build_cnn1d(filters=64, kernel_size=3, dense_units=64, dropout_rate=0.3, lr=1e-3):
    model = Sequential([
        InputLayer(input_shape=(input_dim, 1)),
        Conv1D(filters=filters, kernel_size=kernel_size, activation="relu"),
        Flatten(),
        Dense(dense_units, activation="relu"),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

In [ ]:
cnn_param = {
    "filters": [32, 64, 128],
    "kernel_size": [2, 3, 5],
    "dense_units": [32, 64],
    "dropout_rate": [0.2, 0.3, 0.5],
    "lr": [1e-3, 1e-4, 5e-3],
    "batch_size": [32, 64, 128]
}

In [ ]:
def build_bilstm(units=64, dense_units=64, dropout_rate=0.3, lr=1e-3):
    model = Sequential([
        InputLayer(input_shape=(input_dim, 1)),
        Bidirectional(LSTM(units)),
        Dense(dense_units, activation="relu"),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

In [ ]:
bilstm_param = {
    'units': [32, 64],
    'dense_units': [32, 64],
    'dropout_rate': [0.2, 0.3],
    'lr': [1e-3, 1e-4, 5e-3, 5e-4],
    'batch_size': [32, 64,128]
}

In [ ]:
def build_lstm(units=64, dense_units=64, dropout_rate=0.3, lr=1e-3):
    model = Sequential([
        InputLayer(input_shape=(input_dim, 1)),
        LSTM(units),
        Dense(dense_units, activation="relu"),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

In [ ]:
lstm_param = {
    'units': [32, 64],
    'dense_units': [32, 64],
    'dropout_rate': [0.2, 0.3],
    'lr': [1e-3, 1e-4, 5e-3, 5e-4],
    'batch_size': [32, 64,128],
}

In [ ]:
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    accuracy_score, classification_report
)
from keras.callbacks import Callback, EarlyStopping
from keras.models import model_from_json
import numpy as np
import pickle

class PrintValLossCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Epoch {epoch+1}, Val Loss: {logs['val_loss']:.4f}")

def tune_keras(build_fn, param_dist, X_train, y_train, name):
    n_iter = 5
    param_list = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=SEED))

    best_auc = -np.inf
    best_model = None
    best_params = None

    X_train_reshaped = X_train.reshape(-1, input_dim, 1) if len(X_train.shape) == 2 else X_train
    X_val_reshaped = X_val_pca.reshape(-1, input_dim, 1) if len(X_val_pca.shape) == 2 else X_val_pca
    X_test_reshaped = X_test_pca.reshape(-1, input_dim, 1) if len(X_test_pca.shape) == 2 else X_test_pca

    for i, params in enumerate(param_list):
        print(f"\n====================")
        print(f"🔹 Trial {i+1}/{n_iter}")
        print(f"Training {name} with params: {params}")

        # Tách batch_size
        batch_size = params.pop("batch_size", 32)

        model = build_fn(**params)

        early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        val_loss_logger = PrintValLossCallback()

        history = model.fit(
            X_train_reshaped, y_train,
            epochs=20,
            batch_size=batch_size,
            validation_data=(X_val_reshaped, y_val),
            callbacks=[early_stop, val_loss_logger],
            verbose=0
        )

        print(f"Early stopping at epoch {len(history.history['val_loss'])}")

        print(f"Evaluating {name} on validation set")

        val_probs = model.predict(X_val_reshaped).flatten()
        val_preds = (val_probs >= 0.5).astype(int)

        val_auc = roc_auc_score(y_val, val_probs)
        val_gini = 2 * val_auc - 1
        val_acc = accuracy_score(y_val, val_preds)

        print(f"Test Accuracy: {val_acc}")
        print(f"ROC AUC: {val_auc}")
        print(f"Gini Coefficient: {val_gini}")

        print(classification_report(y_val, val_preds, digits=2))

        if val_auc > best_auc:
            best_auc = val_auc
            best_model = model
            best_params = params.copy()
            best_params["batch_size"] = batch_size

    print(f"\n===== Best {name} Model =====")
    print("Best params:", best_params)
    print("Best Val AUC:", best_auc)

    test_probs = best_model.predict(X_test_reshaped).flatten()
    test_preds = (test_probs >= 0.5).astype(int)

    test_auc = roc_auc_score(y_test, test_probs)
    test_acc = accuracy_score(y_test, test_preds)
    test_gini = 2 * test_auc - 1

    print("\n--- Test Set Metrics ---")
    print(f"Test Accuracy: {test_acc}")
    print(f"ROC AUC: {test_auc}")
    print(f"Gini Coefficient: {test_gini}")
    print(classification_report(y_test, test_preds, digits=2))

    #Lưu mô hình và params dưới dạng pickle để phục vụ XAI
    model_data = {
        "model_json": best_model.to_json(),
        "model_weights": best_model.get_weights(),
        "best_params": best_params,
        "val_auc": best_auc
    }

    with open(f"{name}_xai_model.pkl", "wb") as f:
        pickle.dump(model_data, f)

    print(f"\n Best model and params saved to {name}_xai_model.pkl")

    return best_model, best_params


In [ ]:
tune_keras(build_mlp, mlp_keras_param, X_train_pca, y_train, "MLP_Keras")

In [ ]:
tune_keras(build_cnn1d, cnn_param, X_train_pca, y_train, "CNN1D")

In [ ]:
tune_keras(build_bilstm, bilstm_param, X_train_pca, y_train, "BiLSTM")


🔹 Trial 1/5
Training BiLSTM with params: {'units': 32, 'lr': 0.001, 'dropout_rate': 0.2, 'dense_units': 64, 'batch_size': 128}


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1, Val Loss: 0.6127
Epoch 2, Val Loss: 0.6071
Epoch 3, Val Loss: 0.5830
Epoch 4, Val Loss: 0.5883
Epoch 5, Val Loss: 0.5840
Epoch 6, Val Loss: 0.5707
Epoch 7, Val Loss: 0.5748
Epoch 8, Val Loss: 0.5846
Epoch 9, Val Loss: 0.5757
Epoch 10, Val Loss: 0.5703
Epoch 11, Val Loss: 0.5852
Epoch 12, Val Loss: 0.5782
Epoch 13, Val Loss: 0.5782
Epoch 14, Val Loss: 0.5786
Epoch 15, Val Loss: 0.5726
Early stopping at epoch 15
Evaluating BiLSTM on validation set
1820/1820 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Test Accuracy: 0.6849618635332921
ROC AUC: 0.6962806935358062
Gini Coefficient: 0.3925613870716125
              precision    recall  f1-score   support

           0       0.95      0.69      0.80     53443
           1       0.15      0.59      0.24      4769

    accuracy                           0.68     58212
   macro avg       0.55      0.64      0.52     58212
weighted avg       0.88      0.68      0.76     58212


🔹 Trial 2/5
Training BiLSTM with params: {'units': 64, 'lr': 0.005, 'dr

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1, Val Loss: 0.6101
Epoch 2, Val Loss: 0.5996
Epoch 3, Val Loss: 0.5720
Epoch 4, Val Loss: 0.5531
Epoch 5, Val Loss: 0.5392
Epoch 6, Val Loss: 0.5479
Epoch 7, Val Loss: 0.5575
Epoch 8, Val Loss: 0.5725
Epoch 9, Val Loss: 0.6003
Epoch 10, Val Loss: 0.5862
Early stopping at epoch 10
Evaluating BiLSTM on validation set
1820/1820 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Test Accuracy: 0.7105407819693534
ROC AUC: 0.6845512867563012
Gini Coefficient: 0.36910257351260234
              precision    recall  f1-score   support

           0       0.95      0.73      0.82     53443
           1       0.15      0.53      0.23      4769

    accuracy                           0.71     58212
   macro avg       0.55      0.63      0.53     58212
weighted avg       0.88      0.71      0.77     58212


🔹 Trial 3/5
Training BiLSTM with params: {'units': 64, 'lr': 0.001, 'dropout_rate': 0.3, 'dense_units': 32, 'batch_size': 128}


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


... (2 outputs left out)

Epoch 1, Val Loss: 0.6486
Epoch 2, Val Loss: 0.6205
Epoch 3, Val Loss: 0.6027
Epoch 4, Val Loss: 0.6032
Epoch 5, Val Loss: 0.5968
Epoch 6, Val Loss: 0.5870
Epoch 7, Val Loss: 0.5856
Epoch 8, Val Loss: 0.5802
Epoch 9, Val Loss: 0.5881
Epoch 10, Val Loss: 0.5815
Epoch 11, Val Loss: 0.5788
Epoch 12, Val Loss: 0.5869
Epoch 13, Val Loss: 0.5972
Epoch 14, Val Loss: 0.5929
Epoch 15, Val Loss: 0.5906
Epoch 16, Val Loss: 0.5889
Early stopping at epoch 16
Evaluating BiLSTM on validation set
1820/1820 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Test Accuracy: 0.6841201126915413
ROC AUC: 0.7088248559605957
Gini Coefficient: 0.4176497119211915
              precision    recall  f1-score   support

           0       0.95      0.69      0.80     53443
           1       0.15      0.62      0.24      4769

    accuracy                           0.68     58212
   macro avg       0.55      0.66      0.52     58212
weighted avg       0.89      0.68      0.75     58212


🔹 Trial 5/5
Training BiLSTM with params: {'u

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1, Val Loss: 0.6167
Epoch 2, Val Loss: 0.6108
Epoch 3, Val Loss: 0.6117
Epoch 4, Val Loss: 0.5784
Epoch 5, Val Loss: 0.5804
Epoch 6, Val Loss: 0.5811
Epoch 7, Val Loss: 0.5901
Epoch 8, Val Loss: 0.5936
Epoch 9, Val Loss: 0.5879
Early stopping at epoch 9
Evaluating BiLSTM on validation set
1820/1820 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Test Accuracy: 0.6788634645777503
ROC AUC: 0.6893868700350285
Gini Coefficient: 0.37877374007005704
              precision    recall  f1-score   support

           0       0.95      0.69      0.80     53443
           1       0.14      0.59      0.23      4769

    accuracy                           0.68     58212
   macro avg       0.55      0.64      0.51     58212
weighted avg       0.88      0.68      0.75     58212


===== Best BiLSTM Model =====
Best params: {'units': 32, 'lr': 0.0005, 'dropout_rate': 0.3, 'dense_units': 64, 'batch_size': 128}
Best Val AUC: 0.7088248559605957
1820/1820 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step

--- Test Set Metrics ---
T

(<Sequential name=sequential_3, built=True>,
 {'units': 32,
  'lr': 0.0005,
  'dropout_rate': 0.3,
  'dense_units': 64,
  'batch_size': 128})

In [ ]:
tune_keras(build_lstm, lstm_param, X_train_pca, y_train, "LSTM")